# 02 — Causal inference: estimate the effect

The first two notebooks prepared the data and measured associations. Now answer the causal question:

> **To what extent does applying the backdoor defense, instead of baseline random filtering, change the probability of successful backdoor detection?**

| Variable | Meaning |
|---|---|
| `treatment = 0` | random filtering baseline |
| `treatment = 1` | backdoor defense applied |
| `outcome = 0` | detection failed |
| `outcome = 1` | detection succeeded |

The work happens in four steps, and the order matters:

1. **Build the DAG** — write down what you believe causes what.
2. **Identify** — ask whether, given that DAG, the effect can be computed from observable data at all.
3. **Estimate** — compute the number.
4. **Refute** — stress-test the result.

Nearly all of the thinking is in step 1. Steps 2 to 4 are largely mechanical once the assumptions are fixed.

**Tutorial path:** 00 Data preparation → 01 Correlational analysis → **02 Causal inference**


## 1. The causal estimand

> **Estimand** — the quantity you are trying to learn, defined before you choose a method. It answers "what number do I want?", not "how do I compute it?"

For each unit there are two **potential outcomes**:

- **Y(1)** — would detection succeed if the defense were applied?
- **Y(0)** — would detection succeed under random filtering?

The causal effect for a single unit is the difference `Y(1) − Y(0)`. We can never compute it, because each unit shows only one of the two.

What we *can* target is the average across all units, the **Average Treatment Effect (ATE)**:

**ATE = E[Y(1) − Y(0)]**

where `E[...]` simply means "the average of".

The outcome is binary, so the ATE is a difference between two probabilities and reads naturally in percentage points. An ATE of `0.10` means the defense causes an estimated **10 percentage-point increase** in the detection success rate, on average.

The average is reachable even though no individual difference is: with the right assumptions, the treated group can stand in for what would have happened to the control group, and vice versa.


## 2. Configure the causal analysis

Point the notebook at the dataset from notebook 00 and the worksheet from notebook 01.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from dowhy import CausalModel

from src.causal_graph_ui import CausalGraphBuilder
from src.causal_tutorial_utils import (
    check_dowhy_version,
    load_analysis_data,
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

def default_params():
    return {
        "causal_dataset": "data/causal_data.csv",
        "dag_worksheet_path": "data/dag_worksheet.csv",
        "treatment_column": "treatment",
        "outcome_column": "outcome",
        "covariate_columns": ['code_number_tokens', 'code_complexity', 'code_num_identifiers', 'code_num_strings', 'reviewer_experience', 'rollout_eligibility', 'noise_feature', 'inspection_intensity', 'manual_review_flag'],
        "graph_palette": "colorblind",
        "graph_edge_opacity": 0.35,
        "refuter_simulations": 50,
    }

params = default_params()

print("DoWhy version:", check_dowhy_version("0.14"))
params


## 3. Load the observed data

The model sees the observed treatment, the observed outcome, and the measured covariates — exactly what an analyst would have in a real study.


In [ ]:
(
    analysis_df,
    treatment,
    outcome,
    covariates,
    excluded_covariates,
) = load_analysis_data(params)

print(f"Rows: {len(analysis_df):,}")
print(f"Backdoor-defense prevalence: {analysis_df[treatment].mean():.3f}")
print(f"Observed detection success rate: {analysis_df[outcome].mean():.3f}")
print(f"Covariates available for the graph: {len(covariates)}")
print(f"Excluded constant covariates: {excluded_covariates or 'none'}")

analysis_df.head()


## 4. Review your DAG worksheet

The worksheet records what the data showed, when each variable was measured, and the causal hypotheses you wrote down.

The worksheet is not a causal model; it is evidence and notes. The DAG you build next is where you commit to assumptions.


In [ ]:
worksheet_path = Path(params["dag_worksheet_path"])

if worksheet_path.exists():
    dag_worksheet = pd.read_csv(worksheet_path)
    display(dag_worksheet)
else:
    dag_worksheet = None
    print(
        "DAG worksheet not found. Run "
        "01_correlational_analysis.ipynb first."
    )


## 5. Build the DAG

> **DAG (directed acyclic graph)** — your causal assumptions, drawn. Each variable is a node, and an arrow `A → B` claims that A is a direct cause of B. Leaving out an arrow is also a claim: that there is no direct effect.

The graph starts with the relationship under study:

`treatment → outcome`

That arrow is the hypothesis that applying the defense changes detection success. Everything else you add describes the world around it.

**Add an edge only when you can state a plausible mechanism and a consistent time order.** Two questions before every arrow: *why* would A cause B, and did A come first? "They are correlated" answers neither.

Recall the roles from notebook 01:

- a **confounder** is pre-treatment and causes both, creating bias you need to remove;
- a **mediator** sits on the path `treatment → X → outcome` and carries part of the effect, so adjusting for it hides what you are trying to measure;
- a **collider** is caused by both, and adjusting for it *creates* bias that was not there;
- a variable can predict treatment strongly while having no effect on the outcome at all.

The editor labels structural roles **after** you draw the graph, based on the arrows you chose. It does not infer arrows from correlation — that judgement is yours.


In [ ]:
graph_builder = CausalGraphBuilder(
    data_columns=analysis_df.columns,
    treatment=treatment,
    outcome=outcome,
    covariates=covariates,
    palette=params["graph_palette"],
    edge_opacity=params["graph_edge_opacity"],
)

graph_builder.display()


### Freeze your DAG

When the graph shows assumptions you are willing to defend, click **Use this DAG for analysis**, then run the next cell.

Everything that follows depends on this graph. A different DAG can produce a different estimate from the very same data, which is why the assumptions are written down openly rather than buried inside a modelling choice.


In [ ]:
analysis_dag = graph_builder.get_frozen_graph()

print(
    f"Using DAG with {analysis_dag.number_of_nodes()} nodes "
    f"and {analysis_dag.number_of_edges()} edges."
)


## 6. Create the DoWhy causal model

DoWhy is a causal inference library. It combines four things: the data, which column is the treatment, which column is the outcome, and the DAG you supplied.

The DAG — not the correlation matrix — determines the assumptions DoWhy works from. Give it a different graph and you get a different analysis.


In [ ]:
causal_model = CausalModel(
    data=analysis_df,
    treatment=treatment,
    outcome=outcome,
    graph=analysis_dag,
)

print("DoWhy causal model created from the frozen DAG.")


## 7. Identify the causal effect

**Identification** asks a question that comes before any computation:

> **If this DAG is correct, can the causal effect be written in terms of quantities we can actually observe?**

The answer is not automatic. If an important confounder was never measured, the effect may be impossible to recover from this data no matter how many rows you have, and no statistical method can rescue it. Far better to learn that now than after estimating.

DoWhy usually answers with a **backdoor adjustment**. A **backdoor path** is a non-causal route connecting treatment and outcome through a common cause (`treatment ← X → outcome`); it is exactly what makes the raw comparison misleading. Adjusting for the right set of variables blocks those paths and leaves only the causal effect.

This step settles **what** to estimate. The next one chooses **how**.


In [ ]:
identified_estimand = causal_model.identify_effect(
    proceed_when_unidentifiable=False
)

print(identified_estimand)


## 8. Estimate the ATE

The method used here is **inverse propensity-score weighting**.

> **Propensity score** — a unit's probability of receiving the treatment, given its covariates.

The idea: the treated and control groups are not comparable as they stand, so reweight them until they are. A unit that received a treatment it was unlikely to get counts for more; a unit that received the expected treatment counts for less. After weighting, the two groups resemble each other on their measured covariates, and the remaining difference in outcomes is the estimated effect.

This works only for variables you measured and included. Weighting cannot balance a confounder that is absent from the data.

Read the result in context:

- **positive** — the defense increases detection success on average;
- **negative** — it decreases it;
- **near zero** — little average change, under your assumptions.


In [ ]:
estimate = causal_model.estimate_effect(
    identified_estimand,
    method_name="backdoor.propensity_score_weighting",
    target_units="ate",
    control_value=0,
    treatment_value=1,
    method_params={
        "min_ps_score": 0.05,
        "max_ps_score": 0.95,
        "weighting_scheme": "ips_weight",
    },
)

estimated_ate = float(estimate.value)

print(estimate)
print(f"\nEstimated ATE: {estimated_ate:.4f}")
print(
    "Interpretation: the model estimates a "
    f"{estimated_ate * 100:.1f} percentage-point average change "
    "in detection success when using the backdoor defense rather than "
    "random filtering."
)


## 9. Refute the estimate

**Refutation** checks re-run the analysis on deliberately altered data and ask whether the estimate reacts the way it should.

They are diagnostics, not proof. Passing them does not show the DAG is right or that confounding is gone — only that the estimate did not fail these particular tests.

### Placebo treatment

The treatment column is randomly shuffled, destroying any real relationship with the outcome. A fake treatment should have no effect, so the estimate should now land near zero. If it does not, the analysis is picking up something that is not a treatment effect.


In [ ]:
placebo_refutation = causal_model.refute_estimate(
    identified_estimand,
    estimate,
    method_name="placebo_treatment_refuter",
    placebo_type="permute",
    num_simulations=params["refuter_simulations"],
    random_seed=RANDOM_SEED,
)

print(placebo_refutation)


### Random common cause

A randomly generated variable is added to the model as an extra common cause. It is pure noise, so a sound estimate should barely move. A large shift suggests the estimate is fragile.


In [ ]:
random_common_cause_refutation = causal_model.refute_estimate(
    identified_estimand,
    estimate,
    method_name="random_common_cause",
    num_simulations=params["refuter_simulations"],
    random_seed=RANDOM_SEED,
)

print(random_common_cause_refutation)


## Final interpretation

The three notebooks answered three different questions:

| Notebook | Question | Answer |
|---|---|---|
| 00 | What was observed? | One record per unit, with one treatment and one outcome. |
| 01 | What patterns are visible? | Associations and imbalances, not causal effects. |
| 02 | What is the causal effect? | A DAG, an identified estimand, an estimated ATE, and robustness checks. |

Your estimate is only as good as the DAG behind it, and that DAG came from your reasoning rather than from the data. So the result worth reporting is not the number alone but the number *together with* its assumptions: which variables you treated as confounders, which as mediators or colliders, and why.

A good closing exercise: name the single assumption that, if wrong, would most change your answer.

The lesson of the tutorial: choosing which variables to adjust for is a **causal reasoning problem**, not a correlation-ranking exercise.
